# NEXRAD ≥10 dBZ Echo-Area Timeseries

Compute a 24-hour timeseries of precipitation **echo area above a reflectivity threshold** for any
WSR-88D (NEXRAD) radar, using Py-ART.

**Workflow**
1. List available Level 2 volume scans from NOAA's public cloud mirror.
2. For each volume: read with Py-ART, take the lowest sweep, sum the true ground
   footprint of every gate at or above the threshold.
3. Plot area vs. time and save a CSV.

**Data source.** NOAA NEXRAD Level 2 is public on both AWS (`noaa-nexrad-level2`) and
Google Cloud (`gcp-public-data-nexrad-l2`). The AWS bucket serves object downloads
anonymously but **denies anonymous bucket listing**, so we use the GCS mirror, which
allows anonymous listing. The mirror packages each hour as a `.tar` of individual
volume-scan files and typically lags real time by ~1 day.

> **Requirements:** `arm_pyart`, `numpy`, `pandas`, `matplotlib`, `requests`
> (`conda install -c conda-forge arm_pyart pandas`).


## 1. Parameters

Set the radar site and the day to analyze (UTC).

In [ ]:
# --- user parameters ---
SITE       = "KLOT"          # 4-letter NEXRAD site ID, e.g. KLOT (Chicago), KTLX (Oklahoma City)
END_DATE   = "2026-07-04"    # UTC calendar day that ENDS the 24 h window (YYYY-MM-DD)
THRESHOLD  = 10.0            # reflectivity threshold in dBZ
SWEEP      = 0               # sweep index; 0 = lowest elevation (~0.5 deg)
WINDOW_H   = 24             # length of the window in hours

GCS_BASE = "https://gcp-public-data-nexrad-l2.storage.googleapis.com/"

## 2. Imports and helper functions

In [ ]:
import io, gzip, tarfile, warnings
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pyart

warnings.filterwarnings("ignore")   # silence Py-ART/masked-array chatter


def list_keys(prefix, max_keys=400):
    """Anonymous XML listing of the public GCS NEXRAD bucket."""
    r = requests.get(GCS_BASE, params={"prefix": prefix, "max-keys": str(max_keys)}, timeout=30)
    r.raise_for_status()
    root = ET.fromstring(r.text)
    return sorted(e.text for e in root.iter() if e.tag.endswith("Key"))


def area_above_threshold(radar, thresh=10.0, sweep=0):
    """Ground area (km^2) of gates >= thresh dBZ on one sweep.

    Each gate's footprint is its radial depth (dr) times its azimuthal arc length
    (ground_range * d_azimuth), which corrects for the range-dependent gate size.
    """
    s, e = radar.get_start_end(sweep)
    refl = radar.fields["reflectivity"]["data"][s:e + 1]
    az   = radar.azimuth["data"][s:e + 1]
    rng  = radar.range["data"]                      # metres
    elev = np.deg2rad(float(radar.fixed_angle["data"][sweep]))
    ground = rng[None, :] * np.cos(elev)            # horizontal ground range
    dr   = float(np.median(np.diff(rng)))           # gate depth (m)
    daz  = np.deg2rad(360.0 / len(az))              # azimuthal beamwidth (rad)
    cell_km2 = np.broadcast_to((ground * dr * daz) / 1e6, refl.shape)
    mask = (refl >= thresh) & ~np.ma.getmaskarray(refl)
    return float(cell_km2[mask].sum()), float(np.ma.max(refl))


def volume_time(member_name, site):
    """Parse UTC timestamp from a volume filename like 'KLOT20260704_052243_V06...'."""
    stamp = member_name.split("_")[0].replace(site, "") + member_name.split("_")[1]
    return datetime.strptime(stamp, "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc)

## 3. Which hourly tars cover the window?

We want a 24-hour window ending at the last scan of `END_DATE`. That spans two
calendar days (the tail of the previous day plus `END_DATE`), so we gather the
relevant hourly tars from both.

In [ ]:
end_day   = datetime.strptime(END_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc)
start_day = end_day - timedelta(days=1)

tar_keys = []
for day in (start_day, end_day):
    tar_keys += list_keys(f"{day:%Y/%m/%d}/{SITE}/")
tar_keys = sorted(k for k in tar_keys if k.endswith(".tar"))
print(f"{len(tar_keys)} hourly tars found for {SITE} on {start_day:%Y-%m-%d} and {END_DATE}")
tar_keys[:3], tar_keys[-3:]

## 4. Process every volume scan

Stream each hourly tar into memory, extract each volume, read it with Py-ART, and
compute the echo area. Metadata-only members (`_MDM`) are skipped. This is the slow
step — roughly 1–2 s per volume, ~13 volumes/hour.

In [ ]:
records = []
for i, key in enumerate(tar_keys, 1):
    resp = requests.get(GCS_BASE + key, timeout=180)
    resp.raise_for_status()
    with tarfile.open(fileobj=io.BytesIO(resp.content)) as tar:
        for member in tar.getnames():
            if "_MDM" in member:            # metadata message, not a volume
                continue
            try:
                data = tar.extractfile(member).read()
                raw  = gzip.decompress(data) if member.endswith(".gz") else data
                radar = pyart.io.read_nexrad_archive(io.BytesIO(raw))
                if "reflectivity" not in radar.fields:
                    continue
                area, max_dbz = area_above_threshold(radar, THRESHOLD, SWEEP)
                records.append({"time": volume_time(member, SITE),
                                "area_km2": area, "max_dbz": max_dbz})
            except Exception as exc:
                print(f"  skip {member}: {str(exc)[:60]}")
    print(f"[{i}/{len(tar_keys)}] {key.split('/')[-1]}  cumulative volumes: {len(records)}")

print("done:", len(records), "volumes")

## 5. Assemble the timeseries and trim to the window

In [ ]:
df = pd.DataFrame(records).sort_values("time").drop_duplicates("time").reset_index(drop=True)

t_end   = df["time"].max()
t_start = t_end - pd.Timedelta(hours=WINDOW_H)
win = df[df["time"] >= t_start].copy()

print(f"window: {t_start:%Y-%m-%d %H:%M} -> {t_end:%Y-%m-%d %H:%M} UTC   ({len(win)} scans)")
print(f"area km^2  min={win.area_km2.min():,.0f}  mean={win.area_km2.mean():,.0f}  max={win.area_km2.max():,.0f}")
peak = win.loc[win.area_km2.idxmax()]
print(f"peak: {peak.area_km2:,.0f} km^2 at {peak['time']:%Y-%m-%d %H:%M} UTC (max {peak.max_dbz:.0f} dBZ)")

## 6. Plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.6), constrained_layout=True)
ax.fill_between(win["time"], 0, win["area_km2"] / 1e3, color="#4a7fb5", alpha=0.25)
ax.plot(win["time"], win["area_km2"] / 1e3, color="#1f4e79", lw=1.3, marker="o", ms=2.5)

ax.set_ylabel(f"Echo area \u2265 {THRESHOLD:.0f} dBZ (\u00d710\u00b3 km\u00b2)")
ax.set_xlabel("Time (UTC)")
ax.set_ylim(0, None)
ax.margins(x=0.01)
ax.xaxis.set_major_locator(mdates.HourLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
ax.annotate(f"peak {peak.area_km2/1e3:.0f}k km\u00b2 \u00b7 {peak['time']:%H:%M} UTC",
            xy=(peak["time"], peak.area_km2 / 1e3),
            xytext=(0, 14), textcoords="offset points", ha="center", fontsize=8,
            arrowprops=dict(arrowstyle="->", lw=0.7))
ax.set_title(f"{SITE} \u2014 precipitation echo area (\u2265 {THRESHOLD:.0f} dBZ, "
             f"sweep {SWEEP}) over {WINDOW_H} h", loc="left")

fig.savefig(f"{SITE}_{WINDOW_H}h_area.png", dpi=200)
plt.show()

## 7. Save the data

In [ ]:
out = win.copy()
out["time"] = out["time"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
csv_name = f"{SITE}_{WINDOW_H}h_area.csv"
out.to_csv(csv_name, index=False)
print("wrote", csv_name)
out.head()

## Notes & extensions

- **Threshold.** Raise `THRESHOLD` to ~20 dBZ for stratiform + convective rain, or
  ~35–40 dBZ to isolate convective cores.
- **Real-time data.** The GCS/AWS mirrors lag ~1 day. For the live feed, use the
  real-time AWS bucket (requires authenticated/`--request-payer` access) or the NWS
  feed, then point `list_keys` at that source.
- **Sweep geometry.** Areas use a low-elevation, flat-earth approximation on a single
  sweep — appropriate for the lowest tilt. For a gridded/geolocated product use
  `pyart.map.grid_from_radars` and integrate on the Cartesian grid.
- **Speed.** Reading is CPU-bound in `read_nexrad_archive`. To go faster, parallelize
  the per-volume loop (e.g. `concurrent.futures.ProcessPoolExecutor`) or read only the
  lowest sweep via the `include_fields`/`delay_field_loading` options where available.
